<a id="newton-ode"></a>

$$
\text{抛体运动 (Projectile Motion): 从牛顿第二定律到 ODE} \\ \quad \\
\text{位置 } r(t) = (x(t), y(t)) \text{, 由导数的定义:} \\
\text{速度 (Velocity) } v = \frac{dr}{dt} = \lim_{\Delta t \to 0} \frac{r(t + \Delta t) - r(t)}{\Delta t}, \quad
\text{加速度 (Acceleration) } a = \frac{dv}{dt} = \frac{d^2 r}{dt^2} \\ \quad \\
\text{牛顿第二定律 (Newton's Second Law): } F = m a \implies m \frac{dv}{dt} = F(r, v) \\
\text{写成一阶 ODE 方程组 (状态 } (x, y, v_x, v_y) \text{):} \\
\frac{d}{dt}\begin{bmatrix} x \\ y \\ v_x \\ v_y \end{bmatrix} = \begin{bmatrix} v_x \\ v_y \\ F_x / m \\ F_y / m \end{bmatrix} \\ \quad \\
\text{三种受力情况:} \\
\text{1. 只有重力: } F = (0, -mg) \\
\text{2. 线性阻力 (Linear Drag, 低速, 小物体): } F = (0, -mg) - b \, v \\
\text{3. 二次阻力 (Quadratic Drag, 高速, 空气中的球): } F = (0, -mg) - c \, |v| \, v \quad (\text{方向和速度相反, 大小} \propto |v|^2)
$$

数学基础: [导数的定义](../math_basics/calculus.ipynb#derivative) · [常微分方程](../math_basics/calculus.ipynb#ode) · [高阶 ODE 转一阶方程组](../math_basics/calculus.ipynb#ode-system)

<a id="no-drag"></a>

$$
\text{1. 只有重力: 积分两次得到解析解} \\
\frac{dv_x}{dt} = 0, \quad \frac{dv_y}{dt} = -g, \quad v(0) = (v_0 \cos\theta, \ v_0 \sin\theta), \ r(0) = (0, 0) \\ \quad \\
\text{第一次积分 (速度): } v_x = v_0 \cos\theta, \quad v_y = v_0 \sin\theta - g t \\
\text{第二次积分 (位置): } x = v_0 \cos\theta \, t, \quad y = v_0 \sin\theta \, t - \frac{1}{2} g t^2 \\ \quad \\
\text{轨迹方程 (消去 t, } t = \frac{x}{v_0 \cos\theta} \text{): } y = x \tan\theta - \frac{g x^2}{2 v_0^2 \cos^2\theta} \quad (\text{抛物线 parabola}) \\ \quad \\
\text{飞行时间: } y = t\left(v_0 \sin\theta - \frac{1}{2} g t\right) = 0 \implies T = \frac{2 v_0 \sin\theta}{g} \\
\text{射程 (Range): } R = v_0 \cos\theta \cdot T = \frac{2 v_0^2 \sin\theta \cos\theta}{g} = \frac{v_0^2 \sin 2\theta}{g} \\
\text{最大高度: } v_y = 0 \implies t = \frac{v_0 \sin\theta}{g}, \quad H = \frac{v_0^2 \sin^2\theta}{2g} \\ \quad \\
\text{最佳角度: 对 } \theta \text{ 求导令其为 0} \\
\frac{dR}{d\theta} = \frac{2 v_0^2 \cos 2\theta}{g} = 0 \implies 2\theta = \frac{\pi}{2} \implies \theta = 45^\circ \\
\frac{d^2 R}{d\theta^2} = -\frac{4 v_0^2 \sin 2\theta}{g} < 0 \text{, 是极大值, } R_{\max} = \frac{v_0^2}{g} \\
\text{另外 } \sin 2\theta = \sin(\pi - 2\theta) \text{, 所以 } \theta \text{ 和 } 90^\circ - \theta \text{ 的射程相同}
$$

数学基础: [微积分基本定理 (积分是求导的逆运算)](../math_basics/calculus.ipynb#integral) · [极值的一阶 / 二阶条件](../math_basics/calculus.ipynb#extremum)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

g = 9.81
v0 = 30.0

def rk4_until_ground(accel, state0, h=1e-3, t_max=100):
    # RK4 积分到 y < 0, 最后一步用线性插值求落地点
    def f(s):
        ax, ay = accel(s[2], s[3])
        return np.array([s[2], s[3], ax, ay])
    s = np.array(state0, dtype=float)
    t = 0.0
    path = [s.copy()]
    while t < t_max:
        k1 = f(s)
        k2 = f(s + h / 2 * k1)
        k3 = f(s + h / 2 * k2)
        k4 = f(s + h * k3)
        s_new = s + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        if s_new[1] < 0 and t > 0:
            frac = s[1] / (s[1] - s_new[1])
            s_land = s + frac * (s_new - s)
            path.append(s_land)
            return np.array(path), t + frac * h
        s, t = s_new, t + h
        path.append(s.copy())
    raise RuntimeError('没有落地')

no_drag = lambda vx, vy: (0.0, -g)
for theta_deg in [30, 45, 60]:
    th = np.deg2rad(theta_deg)
    path, T_num = rk4_until_ground(no_drag, [0, 0, v0 * np.cos(th), v0 * np.sin(th)])
    R_formula = v0**2 * np.sin(2 * th) / g
    T_formula = 2 * v0 * np.sin(th) / g
    H_formula = v0**2 * np.sin(th)**2 / (2 * g)
    print(f'theta = {theta_deg}°: 射程 RK4 = {path[-1, 0]:.4f}, 公式 = {R_formula:.4f}; '
          f'时间 RK4 = {T_num:.4f}, 公式 = {T_formula:.4f}; 最大高度 RK4 = {path[:, 1].max():.4f}, 公式 = {H_formula:.4f}')

# 轨迹方程
th = np.deg2rad(60)
path, _ = rk4_until_ground(no_drag, [0, 0, v0 * np.cos(th), v0 * np.sin(th)])
inner = path[:-1]   # 最后一个点是线性插值的落地点, 不在抛物线上
y_parabola = inner[:, 0] * np.tan(th) - g * inner[:, 0]**2 / (2 * v0**2 * np.cos(th)**2)
print(f'RK4 轨迹 == 抛物线方程? {np.allclose(inner[:, 1], y_parabola, atol=1e-8)}')

# 射程对角度: 最大值在 45°
angles = np.linspace(1, 89, 177)
R_all = v0**2 * np.sin(2 * np.deg2rad(angles)) / g
print(f'网格搜索最佳角度 = {angles[np.argmax(R_all)]:.1f}°, R_max = {R_all.max():.4f}, v0^2 / g = {v0**2 / g:.4f}')
print(f'30° 和 60° 的射程相同: {v0**2 * np.sin(np.deg2rad(60)) / g:.4f}, {v0**2 * np.sin(np.deg2rad(120)) / g:.4f}')

<a id="linear-drag"></a>

$$
\text{2. 线性阻力: } m \frac{dv}{dt} = -mg \hat{y} - b v \text{, 令 } k = \frac{b}{m} \\
\frac{dv_x}{dt} = -k v_x, \quad \frac{dv_y}{dt} = -g - k v_y \\ \quad \\
\text{x 方向: 可分离变量 } \implies v_x = v_{x0} e^{-kt} \\
\text{y 方向: 一阶线性 ODE } v_y' + k v_y = -g \text{, 积分因子 } \mu = e^{kt} \\
(v_y e^{kt})' = -g e^{kt} \implies v_y e^{kt} = -\frac{g}{k} e^{kt} + C \implies v_y = -\frac{g}{k} + C e^{-kt} \\
v_y(0) = v_{y0} \implies C = v_{y0} + \frac{g}{k} \implies v_y = -\frac{g}{k} + \left(v_{y0} + \frac{g}{k}\right) e^{-kt} \\ \quad \\
\text{终端速度 (Terminal Velocity): } t \to \infty \text{ 时 } v_y \to -\frac{g}{k} \text{ (重力 = 阻力, } mg = b v_t \text{)} \\ \quad \\
\text{再积分一次得到位置 (} x(0) = y(0) = 0 \text{):} \\
x(t) = \int_0^t v_{x0} e^{-ks} ds = \frac{v_{x0}}{k}\left(1 - e^{-kt}\right) \quad (\text{水平距离有上限 } \frac{v_{x0}}{k}) \\
y(t) = -\frac{g}{k} t + \frac{1}{k}\left(v_{y0} + \frac{g}{k}\right)\left(1 - e^{-kt}\right) \\ \quad \\
\text{检查: } k \to 0 \text{ 时, 用泰勒展开 } 1 - e^{-kt} = kt - \frac{k^2 t^2}{2} + O(k^3) \\
y \approx -\frac{g}{k} t + \frac{1}{k}\left(v_{y0} + \frac{g}{k}\right)\left(kt - \frac{k^2 t^2}{2}\right) = v_{y0} t - \frac{1}{2} g t^2 + O(k) \quad (\text{回到无阻力的结果})
$$

数学基础: [分离变量法](../math_basics/calculus.ipynb#ode-separable) · [一阶线性 ODE 与积分因子](../math_basics/calculus.ipynb#ode-linear) · [泰勒展开](../math_basics/calculus.ipynb#taylor)

In [ ]:
k_lin = 0.3
th = np.deg2rad(45)
vx0, vy0 = v0 * np.cos(th), v0 * np.sin(th)

x_lin = lambda t: vx0 / k_lin * (1 - np.exp(-k_lin * t))
y_lin = lambda t: -g / k_lin * t + (vy0 + g / k_lin) / k_lin * (1 - np.exp(-k_lin * t))
vy_lin = lambda t: -g / k_lin + (vy0 + g / k_lin) * np.exp(-k_lin * t)

t_eval = np.linspace(0, 3, 50)
sol = solve_ivp(lambda t, s: [s[2], s[3], -k_lin * s[2], -g - k_lin * s[3]], [0, 3], [0, 0, vx0, vy0],
                t_eval=t_eval, rtol=1e-11, atol=1e-11)
print(f'解析 x(t) == solve_ivp? {np.allclose(sol.y[0], x_lin(t_eval))}, y(t)? {np.allclose(sol.y[1], y_lin(t_eval))}, v_y(t)? {np.allclose(sol.y[3], vy_lin(t_eval))}')
print(f'终端速度 g / k = {g / k_lin:.4f}, t = 30 时 v_y = {vy_lin(30):.4f}')
print(f'水平距离上限 v_x0 / k = {vx0 / k_lin:.4f}')

# k -> 0 回到无阻力
for k_small in [1e-1, 1e-2, 1e-3]:
    y_k = -g / k_small * 2 + (vy0 + g / k_small) / k_small * (1 - np.exp(-k_small * 2))
    print(f'k = {k_small:<6}: y(2) = {y_k:.6f}, 无阻力 y(2) = {vy0 * 2 - 0.5 * g * 4:.6f}')

<a id="quadratic-drag"></a>

$$
\text{3. 二次阻力: } \frac{dv}{dt} = -g \hat{y} - \frac{c}{m} |v| v, \quad |v| = \sqrt{v_x^2 + v_y^2} \\
\frac{dv_x}{dt} = -\frac{c}{m} \sqrt{v_x^2 + v_y^2} \, v_x, \quad \frac{dv_y}{dt} = -g - \frac{c}{m} \sqrt{v_x^2 + v_y^2} \, v_y \\ \quad \\
\text{x 和 y 两个方程通过 } |v| \text{ 耦合在一起 (非线性 ODE), 没有初等函数的解析解, 只能数值求解 (RK4)} \\ \quad \\
\text{只有竖直下落时可以解析求终端速度: } mg = c v_t^2 \implies v_t = \sqrt{\frac{mg}{c}} \\ \quad \\
\text{有阻力时最佳角度 < 45°: 阻力让水平速度一直衰减, 飞行时间越长损失越多} \\
\text{角度小一点 → 飞行时间更短, 水平速度保留得更多, 两者权衡的结果是最佳角度变小} \\
\text{(这时 } \frac{dR}{d\theta} = 0 \text{ 没有解析式, 用数值优化 } \max_\theta R(\theta) \text{)}
$$

In [ ]:
c_over_m = 0.01

def quad_drag(vx, vy):
    speed = np.sqrt(vx**2 + vy**2)
    return -c_over_m * speed * vx, -g - c_over_m * speed * vy

def hit_ground(t, s):
    return s[1]
hit_ground.terminal = True
hit_ground.direction = -1

def rhs_quad(t, s):
    ax, ay = quad_drag(s[2], s[3])
    return [s[2], s[3], ax, ay]

def range_quad(theta_deg):
    th = np.deg2rad(theta_deg)
    path, _ = rk4_until_ground(quad_drag, [0, 0, v0 * np.cos(th), v0 * np.sin(th)], h=2e-3)
    return path[-1, 0]

th = np.deg2rad(45)
path_q, T_q = rk4_until_ground(quad_drag, [0, 0, v0 * np.cos(th), v0 * np.sin(th)])
sol_q = solve_ivp(rhs_quad, [0, 20], [0, 0, v0 * np.cos(th), v0 * np.sin(th)], events=hit_ground, rtol=1e-10, atol=1e-10)
print(f'二次阻力 45°: RK4 射程 = {path_q[-1, 0]:.6f}, 时间 = {T_q:.6f}; solve_ivp 射程 = {sol_q.y_events[0][0][0]:.6f}, 时间 = {sol_q.t_events[0][0]:.6f}')

# 竖直下落的终端速度
sol_fall = solve_ivp(rhs_quad, [0, 30], [0, 1000, 0, 0], rtol=1e-10)
print(f'竖直下落 30 s 后速度 = {sol_fall.y[3, -1]:.4f}, 终端速度 sqrt(g / (c/m)) = {np.sqrt(g / c_over_m):.4f}')

# 最佳角度
res = minimize_scalar(lambda a: -range_quad(a), bounds=(20, 50), method='bounded', options={'xatol': 1e-3})
print(f'二次阻力的最佳角度 = {res.x:.2f}° (< 45°), 最大射程 = {-res.fun:.4f}, 45° 的射程 = {range_quad(45):.4f}')

angles = np.arange(10, 81, 5)
plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
for drag_name, accel in [('no drag', no_drag), ('linear drag', lambda vx, vy: (-k_lin * vx, -g - k_lin * vy)), ('quadratic drag', quad_drag)]:
    p, _ = rk4_until_ground(accel, [0, 0, v0 * np.cos(th), v0 * np.sin(th)])
    plt.plot(p[:, 0], p[:, 1], label=drag_name)
plt.xlabel('x (m)')
plt.ylabel('y (m)')
plt.title('45 degree launch')
plt.subplot(1, 2, 2)
plt.plot(angles, v0**2 * np.sin(2 * np.deg2rad(angles)) / g, 'o-', label='no drag')
plt.plot(angles, [range_quad(a) for a in angles], 's-', label='quadratic drag')
plt.axvline(res.x, color='r', linestyle='--', label=f'optimum {res.x:.1f} deg')
plt.xlabel('theta (deg)')
plt.ylabel('range (m)')
plot_nicely()
plt.show()